# CDC PLACES ZCTA Data → Python EDA

This notebook:
1. **Loads** CDC PLACES data at the **ZCTA** (ZIP Code Tabulation Area) level using the **CDCPLACES R package**.
2. **Converts** the data to a **one-ZCTA-per-row** format (analysis-ready) in Python.
3. Runs **basic exploratory data analysis** (EDA) in Python.

Requires: R with `CDCPLACES` installed, and Python with `rpy2`, `pandas`, `matplotlib`, `seaborn`.

In [ ]:
# Python setup: rpy2 (R from Python), pandas, and EDA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Install rpy2 if missing (run once). If you still get ModuleNotFoundError, restart kernel and run again.
#%pip install rpy2 -q

# R <-> pandas conversion (modern rpy2 API; avoid deprecated activate())
from rpy2.robjects import pandas2ri, default_converter
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

# Use this when converting R data.frames to pandas (see next cell):
#   with localconverter(default_converter + pandas2ri.converter):
#       df = pandas2ri.rpy2py(r_object)

print("Python and rpy2 ready.")

Note: you may need to restart the kernel to use updated packages.
Python and rpy2 ready.


## 1. Load ZCTA data via R (CDCPLACES)

We use the CDCPLACES R package from Python via `rpy2`. Adjust `states` and `measures` to control which ZCTAs and health measures are pulled.

In [9]:
# Load the CDCPLACES R package and fetch ZCTA-level data
import os
import tempfile
from rpy2.robjects import r
from rpy2.robjects.vectors import StrVector
from rpy2.robjects.packages import importr

# Ensure CDCPLACES is loaded in R (install in R first if needed: install.packages("CDCPLACES"))
cdcplaces = importr("CDCPLACES")

# Configuration: state(s) and measure(s) to pull
# For ZCTA, use state to filter (no county filter). Use measure = NULL to get all measures (slower).
states = ["CA"]  # e.g. ["CA"], ["CA", "NY"], or one state for faster runs
measures = ["CASTHMA", "COPD", "OBESITY", "DIABETES"]  # example; use None for all
release = "2023"

# Build R call: get_places(geography = "zcta", state = ..., measure = ..., release = ..., geometry = FALSE)
r.assign("states_r", StrVector(states))
r.assign("measures_r", StrVector(measures))
r.assign("release_r", release)

# get_places(geography = "zcta", state = c("CA"), measure = c("CASTHMA", "COPD", ...), release = "2023", geometry = FALSE)
r('places_zcta <- get_places(geography = "zcta", state = states_r, measure = measures_r, release = release_r, geometry = FALSE)')

# Avoid rpy2 conversion issues (coordinates/numpy): have R write CSV, Python reads it
tmp_csv = os.path.join(tempfile.gettempdir(), "places_zcta_r.csv")
r.assign("tmp_path", tmp_csv)
r('''
if ("sf" %in% class(places_zcta)) {
  if (requireNamespace("sf", quietly = TRUE)) places_zcta <- sf::st_drop_geometry(places_zcta)
}
# Keep only atomic columns (no list/geometry) so write.csv works
atomic <- sapply(places_zcta, is.atomic)
places_zcta <- places_zcta[, atomic, drop = FALSE]
write.csv(places_zcta, tmp_path, row.names = FALSE)
''')

places_long = pd.read_csv(tmp_csv)
try:
    os.remove(tmp_csv)
except OSError:
    pass

print("Raw data (long: one row per ZCTA × measure):")
print(places_long.shape)
places_long.head(10)

R callback write-console: Error in utils::write.table(places_zcta, tmp_path, row.names = FALSE,  : 
  unimplemented type 'list' in 'EncodeElement'
  


RRuntimeError: Error in utils::write.table(places_zcta, tmp_path, row.names = FALSE,  : 
  unimplemented type 'list' in 'EncodeElement'


## 2. Reshape to one ZCTA per row

The API returns one row per (ZCTA, measure). We pivot so each measure becomes a column and each row is a single ZCTA.

In [ ]:
# Inspect column names (CDCPLACES may use slightly different names by release)
print("Columns:", list(places_long.columns))

# Detect column roles: ZCTA/geo id, measure id, and data value
def first_in(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

id_col = first_in(places_long.columns, ["LocationName", "ZCTA", "zcta", "locationname", "ZCTA5"])
measure_col = first_in(places_long.columns, ["MeasureId", "measureid", "Measure"])
value_col = first_in(places_long.columns, ["Data_Value", "data_value", "DataValue"])

# Fallback: if names differ, infer from position (long format often: geo, measure, value)
if id_col is None or measure_col is None or value_col is None:
    cols = list(places_long.columns)
    if id_col is None and len(cols) >= 1:
        id_col = cols[0]
    if measure_col is None and len(cols) >= 2:
        measure_col = cols[1]
    if value_col is None and len(cols) >= 3:
        value_col = cols[2]

print(f"Using: id_col={id_col!r}, measure_col={measure_col!r}, value_col={value_col!r}")

In [ ]:
# Pivot to one row per ZCTA; each measure becomes a column
# Keep state/geo identifiers if present for merging later
index_cols = [id_col]
if "StateAbbr" in places_long.columns:
    index_cols.insert(0, "StateAbbr")
elif "state_abbr" in places_long.columns:
    index_cols.insert(0, "state_abbr")

# Ensure we only use columns that exist
index_cols = [c for c in index_cols if c in places_long.columns]

places_wide = places_long.pivot_table(
    index=index_cols,
    columns=measure_col,
    values=value_col,
    aggfunc="first"  # one value per (ZCTA, measure)
).reset_index()

# Rename index-like column to a clear name for analysis
places_wide = places_wide.rename(columns={id_col: "ZCTA"}) if id_col != "ZCTA" else places_wide

print("One ZCTA per row (analysis-ready):")
print(places_wide.shape)
places_wide.head(10)

## 3. Basic EDA

Summary statistics, missing values, and distributions.

In [ ]:
# Summary: shape, dtypes, missing
print("Shape:", places_wide.shape)
print("\nDtypes:")
print(places_wide.dtypes)
print("\nMissing values per column:")
print(places_wide.isna().sum())
print("\nBasic describe (numeric):")
places_wide.describe()

In [ ]:
# Distributions: histograms for each measure (numeric columns only)
measure_cols = [c for c in places_wide.columns if c not in ["ZCTA", "StateAbbr", "state_abbr"] and places_wide[c].dtype in ["float64", "int64"]]
if not measure_cols:
    measure_cols = [c for c in places_wide.columns if places_wide[c].dtype in ["float64", "int64"]]

n = len(measure_cols)
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(4 * ((n + 1) // 2), 4 * 2))
if n == 1:
    axes = [axes]
else:
    axes = axes.flatten()
for i, col in enumerate(measure_cols):
    axes[i].hist(places_wide[col].dropna(), bins=30, edgecolor="black", alpha=0.7)
    axes[i].set_title(col)
    axes[i].set_xlabel("Value")
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.suptitle("Distribution of health measures (per ZCTA)", y=1.02)
plt.show()

In [ ]:
# Correlation matrix of health measures (if multiple numeric columns)
if len(measure_cols) >= 2:
    corr = places_wide[measure_cols].corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, square=True)
    plt.title("Correlation between health measures (ZCTA-level)")
    plt.tight_layout()
    plt.show()
else:
    print("Need at least 2 numeric measure columns for correlation plot.")

In [ ]:
# Optional: save analysis-ready data for use in other scripts
# out_path = "places_zcta_one_per_row.csv"
# places_wide.to_csv(out_path, index=False)
# print(f"Saved to {out_path}")

# Final view: analysis-ready dataframe (one ZCTA per row)
places_wide

---

**Next steps:** Use `places_wide` in other analyses (e.g. merge with pesticide exposure by ZCTA). Uncomment the save cell above to export `places_zcta_one_per_row.csv`. To pull more states or measures, edit `states` and `measures` in the first data-load cell and re-run.